# 📊 Evaluación de Modelos Entrenados
**Detección de Parkinson en voz — TFM**

---
Este notebook carga cualquier modelo guardado (`.pt`) y genera:

- ✅ **Métricas completas** — Accuracy, Sensibilidad, Especificidad, F1, MCC, AUC-ROC
- 📈 **Curva ROC** con AUC
- 🔲 **Matriz de Confusión** normalizada
- 📋 **Tabla comparativa** de todos los experimentos registrados

**Uso:** Solo cambia `MODEL_PT_PATH` en la celda de configuración.

## ⚙️ CONFIGURACION

In [ ]:
# Ruta al archivo .pt del modelo que quieres evaluar
# (generado automáticamente por los notebooks de entrenamiento)
MODEL_PT_PATH = "../experiments/saved_models/XXXXXX_baseline_cnn1d_neurovoz_fold1.pt"

# Conjunto de datos sobre el que evaluar:
#   "test"  → Test set del fold con el que se entrenó (lo más correcto)
#   "val"   → Validation set
EVAL_SPLIT = "test"

print(f"Evaluando: {MODEL_PT_PATH}")
print(f"Split: {EVAL_SPLIT}")

## 1. Imports

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

from src.models.cnn_1d import CNN_MODELS
from src.models.tcn import TCN_MODELS
from src.models.transformer_audio import TRANSFORMER_MODELS
from src.training.audio_dataset import get_dataloaders
from src.training.trainer import Trainer
from src.evaluation.metrics import calculate_metrics

ALL_MODELS = {**CNN_MODELS, **TCN_MODELS, **TRANSFORMER_MODELS}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

## 2. Cargar modelo

In [ ]:
checkpoint = torch.load(MODEL_PT_PATH, map_location=DEVICE)

model_name  = checkpoint["model_name"]
hyperparams = checkpoint["hyperparams"]
dataset     = checkpoint["dataset"]
fold        = checkpoint["fold"]

print(f"Modelo cargado: {model_name}")
print(f"Dataset: {dataset} | Fold: {fold}")
print(f"Hiperparámetros: {hyperparams}")

# Reconstruir el modelo con los mismos hiperparámetros
ModelClass = ALL_MODELS[model_name]

if model_name == "baseline_cnn1d":
    model = ModelClass(n_filters=hyperparams.get("n_filters", [32,64,128]),
                       kernel_size=hyperparams.get("kernel_size", 7),
                       dropout=hyperparams.get("dropout", 0.3), num_classes=2)
elif model_name == "residual_cnn1d":
    model = ModelClass(n_filters=hyperparams.get("n_filters", [32,64,128,256]),
                       kernel_size=hyperparams.get("kernel_size", 3),
                       n_res_blocks=hyperparams.get("n_res_blocks", 2),
                       dropout=hyperparams.get("dropout", 0.2), num_classes=2)
elif model_name == "dilated_cnn1d":
    model = ModelClass(n_channels=hyperparams.get("n_channels", 64),
                       kernel_size=hyperparams.get("kernel_size", 3),
                       dilations=hyperparams.get("dilations", [1,2,4,8,16]),
                       dropout=hyperparams.get("dropout", 0.2), num_classes=2)
elif model_name == "tcn":
    model = ModelClass(num_channels=hyperparams.get("num_channels", [64,64,64,64]),
                       kernel_size=hyperparams.get("kernel_size", 3),
                       dropout=hyperparams.get("dropout", 0.2), num_classes=2)
elif model_name == "wavenet_like":
    model = ModelClass(residual_channels=hyperparams.get("residual_channels", 32),
                       skip_channels=hyperparams.get("skip_channels", 64),
                       n_layers=hyperparams.get("n_layers", 8),
                       kernel_size=hyperparams.get("kernel_size", 2), num_classes=2)
elif model_name == "temporal_transformer":
    model = ModelClass(patch_size=hyperparams.get("patch_size", 64),
                       d_model=hyperparams.get("d_model", 128),
                       nhead=hyperparams.get("nhead", 4),
                       num_layers=hyperparams.get("num_layers", 4),
                       dim_feedforward=hyperparams.get("dim_feedforward", 256),
                       dropout=hyperparams.get("dropout", 0.1), num_classes=2)
elif model_name == "cnn_transformer_hybrid":
    model = ModelClass(cnn_channels=hyperparams.get("cnn_channels", [32,64,128]),
                       d_model=hyperparams.get("d_model", 128),
                       nhead=hyperparams.get("nhead", 4),
                       num_layers=hyperparams.get("num_layers", 3),
                       dim_feedforward=hyperparams.get("dim_feedforward", 256),
                       dropout=hyperparams.get("dropout", 0.1), num_classes=2)

model.load_state_dict(checkpoint["state_dict"])
model = model.to(DEVICE)
model.eval()
print(f"\nModelo listo. Parámetros: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Cargar datos del split correspondiente

In [ ]:
SPLITS_JSON = os.path.join(PROJECT_ROOT, "data", "data_splits.json")
DATA_ROOT   = os.path.join(PROJECT_ROOT, "data", "processed")

train_loader, val_loader, test_loader = get_dataloaders(
    splits_json=SPLITS_JSON, dataset_name=dataset,
    fold_index=fold - 1,   # fold guardado como 1..K, el índice es 0..K-1
    data_root=DATA_ROOT, batch_size=32, augment_train=False,
)

eval_loader = test_loader if EVAL_SPLIT == "test" else val_loader
print(f"Split a evaluar: {EVAL_SPLIT}")

## 4. Inferencia

In [ ]:
dummy_optim = torch.optim.Adam(model.parameters())
dummy_crit  = torch.nn.CrossEntropyLoss()
trainer = Trainer(model, dummy_optim, dummy_crit, DEVICE)

_, all_preds, all_probs, all_labels = trainer.evaluate(eval_loader)

print(f"Total muestras evaluadas: {len(all_labels)}")
print(f"Parkinson (1): {sum(all_labels)} | Control (0): {sum(l == 0 for l in all_labels)}")

## 5. Métricas completas

In [ ]:
metrics = calculate_metrics(all_labels, all_preds, all_probs)

print("\n" + "═"*55)
print(f"  RESULTADOS: {model_name.upper()}")
print(f"  Dataset: {dataset.upper()} | Fold: {fold} | Split: {EVAL_SPLIT.upper()}")
print("═"*55)
metrics_display = {
    "Accuracy":       metrics["accuracy"],
    "Sensibilidad":   metrics["sensibilidad"],
    "Especificidad":  metrics["especificidad"],
    "F1 Score":       metrics["f1"],
    "MCC":            metrics["mcc"],
    "AUC-ROC":        metrics["auc_roc"],
}
for k, v in metrics_display.items():
    stars = "★" * min(5, int(v * 5)) if not np.isnan(v) else ""
    print(f"  {k:<22}: {v:.4f}  {stars}")

## 6. Curva ROC

In [ ]:
if len(np.unique(all_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    auc = roc_auc_score(all_labels, all_probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="crimson", lw=2.5, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="navy", lw=1.5, label="Clasificador aleatorio")

    # Punto óptimo (máximo Youden = sensibilidad + especificidad - 1)
    youden = tpr - fpr
    best_idx = np.argmax(youden)
    ax.scatter(fpr[best_idx], tpr[best_idx], color="gold", s=120, zorder=5,
               label=f"Umbral óptimo = {thresholds[best_idx]:.3f}")

    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel("Tasa Falsos Positivos (1 - Especificidad)", fontsize=12)
    ax.set_ylabel("Tasa Verdaderos Positivos (Sensibilidad)", fontsize=12)
    ax.set_title(f"Curva ROC — {model_name.upper()}\n{dataset.upper()} Fold {fold}", fontsize=13)
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    # Guardar la figura
    roc_save_dir = os.path.join(PROJECT_ROOT, "experiments", "figures")
    os.makedirs(roc_save_dir, exist_ok=True)
    roc_path = os.path.join(roc_save_dir, f"roc_{model_name}_{dataset}_fold{fold}.png")
    plt.savefig(roc_path, dpi=150)
    print(f"Curva ROC guardada: {roc_path}")
    plt.show()
else:
    print("No se puede calcular ROC: solo hay una clase en el conjunto.")

## 7. Matriz de Confusión

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = confusion_matrix(all_labels, all_preds, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp1 = ConfusionMatrixDisplay(cm, display_labels=["Control (0)", "Parkinson (1)"])
disp1.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusión — Valores Absolutos", fontsize=12)

disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=["Control (0)", "Parkinson (1)"])
disp2.plot(ax=axes[1], colorbar=False, cmap="Oranges", values_format=".2f")
axes[1].set_title("Matriz de Confusión — Normalizada", fontsize=12)

plt.suptitle(f"{model_name.upper()} — {dataset.upper()} Fold {fold}", fontsize=13, y=1.02)
plt.tight_layout()

cm_path = os.path.join(PROJECT_ROOT, "experiments", "figures",
                       f"cm_{model_name}_{dataset}_fold{fold}.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
print(f"Matriz de confusión guardada: {cm_path}")
plt.show()

## 8. Tabla comparativa de todos los experimentos

In [ ]:
LOG_CSV = os.path.join(PROJECT_ROOT, "experiments", "experiments_log.csv")

if os.path.exists(LOG_CSV):
    df = pd.read_csv(LOG_CSV)

    # Columnas relevantes para comparación
    cols_show = ["run_id", "model_name", "dataset", "fold",
                 "accuracy", "sensibilidad", "especificidad", "f1", "mcc", "auc_roc",
                 "epochs_trained", "notes"]
    df_show = df[[c for c in cols_show if c in df.columns]]

    # Ordenar por AUC-ROC descendente
    if "auc_roc" in df_show.columns:
        df_show = df_show.sort_values("auc_roc", ascending=False)

    # Resaltar el mejor por AUC
    def highlight_best(s):
        if s.name in ["accuracy", "sensibilidad", "especificidad", "f1", "mcc", "auc_roc"]:
            is_max = s == s.max()
            return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_max]
        return ["" for _ in s]

    display(df_show.style.apply(highlight_best).format(
        {c: "{:.4f}" for c in ["accuracy","sensibilidad","especificidad","f1","mcc","auc_roc"]
         if c in df_show.columns}
    ))
else:
    print(f"No se encontró el log en: {LOG_CSV}")
    print("Ejecuta primero al menos un notebook de entrenamiento.")